### imports

In [7]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from my_ml.datasets import load_spambase

KeyboardInterrupt: 

In [ ]:
from my_ml.linear_models.regression import LinearRegression
from my_ml.preprocessing import Data_split,KFold

In [ ]:
from my_ml.trees import DecisionTree_Classification, DecisionTree_Regression,RandomForestClassification

In [ ]:
import numpy as np

def confusion_matrix(y_true, y_pred):
    classes = np.unique(np.concatenate((y_true, y_pred)))
    n = len(classes)
    cm = np.zeros((n, n), dtype=int)

    class_to_idx = {cls: i for i, cls in enumerate(classes)}

    for t, p in zip(y_true, y_pred):
        cm[class_to_idx[t], class_to_idx[p]] += 1

    return cm


### Decision Trees

In [ ]:
X,y = load_spambase()

In [ ]:
(X_train,y_train),(X_test,y_test),(X_val,y_val) = Data_split(X,y,[0.5,0.25],True,True)

In [ ]:
fold = KFold(k_split=5,shuffle=True)
splits = list(fold.split(X))

In [ ]:
X, y = fetch_california_housing(return_X_y=True)

In [ ]:
model = DecisionTree_Regression(
   threshold=0.01,
   max_depth=5,
   min_samples_split=80,
   min_samples_leaf=40,
)


In [ ]:
model.fit(X,y)

In [ ]:
y_pred = model.predict(X)

### PCA test

In [ ]:
from my_ml.decomposition import PCA,KPCA

In [ ]:
X,y = load_spambase()

In [ ]:
decomp = KPCA(n_components=2,gamma=1.0)
decomp.fit(X)

In [ ]:
X_trans = decomp.transform(X)

In [ ]:
X_trans

array([[4.97802848e-06, 4.97938295e-06],
       [3.24702633e-06, 3.22263730e-06],
       [3.24626258e-06, 3.22186819e-06],
       ...,
       [3.24894779e-06, 3.22457240e-06],
       [3.50018160e-06, 3.47790223e-06],
       [3.24672763e-06, 3.22233660e-06]], shape=(4601, 2))

### tensor

In [1]:
import numpy as np
import my_ml
from my_ml.mytorch import tensor

In [1]:
import numpy as np
import my_ml
from my_ml.mytorch import tensor

# ---------------- Numerical gradient helper ---------------- #

def numerical_grad(f, x, eps=1e-6):
    grad = np.zeros_like(x.data)
    it = np.nditer(x.data, flags=['multi_index'], op_flags=['readwrite'])

    while not it.finished:
        idx = it.multi_index

        old = x.data[idx]

        x.data[idx] = old + eps
        f_pos = f().data.sum()

        x.data[idx] = old - eps
        f_neg = f().data.sum()

        x.data[idx] = old
        grad[idx] = (f_pos - f_neg) / (2 * eps)

        it.iternext()
    return grad


# ---------------- Test 1: Basic Ops ---------------- #

def test_basic_ops(tensor):
    print("\n=== TEST 1: BASIC OPS ===")

    a = tensor(np.random.randn(3, 4))
    b = tensor(np.random.randn(3, 4))
    c = tensor(np.random.randn(1, 4))   # broadcasting

    out = (a + b) * a
    out = out.relu() + c.sigmoid()

    s = out.data.sum()
    out.backward()

    print("Output sum:", s)
    print("Grad(a):", a.grad)
    print("Grad(b):", b.grad)
    print("Grad(c):", c.grad)


# ---------------- Test 2: Matmul + activations ---------------- #

def test_matmul(tensor):
    print("\n=== TEST 2: MATMUL ===")

    X = tensor(np.random.randn(5, 10))
    W = tensor(np.random.randn(10, 6))
    b = tensor(np.random.randn(1, 6))

    out = X @ W
    out = out + b
    out = out.sigmoid().relu()

    s = out.data.sum()
    out.backward()

    print("Loss:", s)
    print("Grad(W):", W.grad)
    print("Grad(b):", b.grad)
    print("Grad(X):", X.grad)


# ---------------- Test 3: Large MLP ---------------- #

def test_large_mlp(tensor):
    print("\n=== TEST 3: 3-LAYER MLP ===")

    np.random.seed(0)

    X = tensor(np.random.randn(32, 20))
    y = tensor(np.random.randn(32, 10))

    W1 = tensor(np.random.randn(20, 64))
    b1 = tensor(np.random.randn(1, 64))

    W2 = tensor(np.random.randn(64, 64))
    b2 = tensor(np.random.randn(1, 64))

    W3 = tensor(np.random.randn(64, 10))
    b3 = tensor(np.random.randn(1, 10))

    # forward
    h1 = (X @ W1 + b1).relu()
    h2 = (h1 @ W2 + b2).sigmoid()
    out = h2 @ W3 + b3

    loss = ((out - y) * (out - y)).data.mean()

    (out - y).backward()

    print("LOSS:", loss)
    print("Grad(W1) sum:", W1.grad.sum())
    print("Grad(W2) sum:", W2.grad.sum())
    print("Grad(W3) sum:", W3.grad.sum())
    print("Grad(X)  sum:", X.grad.sum())


# ---------------- Test 4: Gradient check ---------------- #

def test_gradcheck(tensor):
    print("\n=== TEST 4: GRAD CHECK ===")

    np.random.seed(1)
    x = tensor(np.random.randn(4, 5))

    def f():
        return (x * x + x).sigmoid().relu()

    y = f()
    y.backward()

    grad_analytical = x.grad.copy()
    grad_numeric = numerical_grad(f, x)

    diff = np.abs(grad_analytical - grad_numeric).mean()

    print("Mean absolute gradient diff:", diff)
    print("Should be < 1e-4")


# ---------------- RUN ALL ---------------- #

def run_all_tests(tensor):
    test_basic_ops(tensor)
    test_matmul(tensor)
    test_large_mlp(tensor)
    test_gradcheck(tensor)


In [2]:
run_all_tests(tensor)


=== TEST 1: BASIC OPS ===
Output sum: 29.53228251767966
Grad(a): [[ 0.          6.89547296  2.16118986  3.21362863]
 [ 1.61145031 -1.5851072   1.03011536 -5.02732339]
 [ 0.86842903  3.01409545  1.87006996  0.        ]]
Grad(b): [[ 0.          2.84503929  0.41998837  1.80044404]
 [ 0.41461121 -0.80424414  0.33949125 -1.07646982]
 [ 0.06814229  1.47121461  0.79365194  0.        ]]
Grad(c): [[0.74823166 0.55451631 0.34469055 0.67726104]]

=== TEST 2: MATMUL ===
Loss: 13.601810708161674
Grad(W): [[-2.39788410e-01 -1.77897398e-01 -1.04483592e-01 -7.63137011e-02
  -6.28546170e-02 -1.69062906e-01]
 [ 8.16806927e-02  2.18716866e-01  2.49444077e-01  6.31315984e-04
   8.40539924e-02  8.41041883e-02]
 [-5.59434560e-02  2.45066132e-01 -2.35752350e-02  2.69143653e-02
   3.24821861e-01 -9.10418080e-03]
 [-3.98215014e-02  4.28794444e-01  1.22400247e-01  8.88323720e-02
   4.30946244e-01 -8.69082455e-03]
 [-1.78283724e-02 -4.35411148e-01 -6.03889702e-01 -7.83453391e-02
   1.47155136e-02 -8.27989020e-0